# LST CVAE-U-Net Skip 학습 v2.5

- Prototype CVAE 구조를 유지하되, condition encoder의 중간 해상도 feature를 decoder에 skip으로 연결합니다.
- posterior encoder는 `condition + target + meta`에서 `mu/logvar`만 만들고, skip에는 절대 쓰지 않습니다. target leakage 방지 목적입니다.
- 입력 condition: 지형, 알베도, AWS 기후장, EGIS LV1 one-hot.
- 타깃: `lst_k_filled` 1채널.
- 저장: `latest.pt`, `best_val.pt`, `best.pt`, `best_balanced.pt`, `checkpoints/epoch_XXX.pt`, `checkpoint_summary.csv`.


In [ ]:
# 필요하면 먼저 실행
%pip install -q torch numpy pandas matplotlib tqdm

from __future__ import annotations

from pathlib import Path
import json
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm


In [ ]:
SEED = 20260522
BATCH_SIZE = 4
EPOCHS = 50
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
BETA = 10 ** -3.5
KL_WARMUP_EPOCHS = 10
LATENT_DIM = 64
WIDTH = 32
PATIENCE = 12
NUM_WORKERS = 0
BALANCED_MIN_KL = 0.004
SAVE_EVERY_EPOCH = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name in {"final", "attempt1", "attempt2"}:
    PROJECT_DIR = PROJECT_DIR.parent
DATA_DIR = PROJECT_DIR / "attempt1" / "lst_dataset"
SAMPLE_DIR = DATA_DIR / "samples"
RUN_DIR = DATA_DIR / "runs" / "lst_cvae_unet_skip_v25"
RUN_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR:", DATA_DIR)
print("RUN_DIR:", RUN_DIR)
print("DEVICE:", DEVICE)


In [ ]:
train_paths = sorted((SAMPLE_DIR / "train").glob("*.npz"))
val_paths = sorted((SAMPLE_DIR / "val").glob("*.npz"))
test_paths = sorted((SAMPLE_DIR / "test").glob("*.npz"))
all_paths = train_paths + val_paths + test_paths

print("samples train/val/test:", len(train_paths), len(val_paths), len(test_paths))
print("samples total:", len(all_paths))
if not train_paths:
    raise FileNotFoundError("train samples are empty. Run sample generation first.")


In [ ]:
def npz_scalar(value):
    return value.item() if hasattr(value, "item") else value


def read_channels(path):
    with np.load(path, allow_pickle=True) as z:
        condition_channels = [str(x) for x in z["condition_channels"]]
        target_channels = [str(x) for x in z["target_channels"]]
        meta_channels = [str(x) for x in z["meta_channels"]]
    return condition_channels, target_channels, meta_channels


CONDITION_CHANNELS, TARGET_CHANNELS, META_CHANNELS = read_channels(train_paths[0])
print("condition channels:", CONDITION_CHANNELS)
print("target channels:", TARGET_CHANNELS)
print("meta channels:", META_CHANNELS)

banned_tokens = ["air", "o3", "pm10", "pm25", "ndvi", "pressure", "press", "pa", "nearest_lst"]
bad_channels = [name for name in CONDITION_CHANNELS + TARGET_CHANNELS if any(token in name.lower() for token in banned_tokens)]
if bad_channels:
    raise RuntimeError(f"banned channels found: {bad_channels}")
if TARGET_CHANNELS != ["lst_k_filled"]:
    raise RuntimeError(f"target must be ['lst_k_filled'], got {TARGET_CHANNELS}")

required_condition = [
    "elevation", "slope", "aspect_sin", "aspect_cos", "hillshade_landsat",
    "albedo", "avg_temp", "wind_u", "wind_v", "rain_1h", "humidity",
]
missing_required = [name for name in required_condition if name not in CONDITION_CHANNELS]
if missing_required:
    raise RuntimeError(f"missing required condition channels: {missing_required}")

egi_channels = [name for name in CONDITION_CHANNELS if name.startswith("egis_cat_")]
print("EGIS categorical channels:", egi_channels)
if "egis_cat_0_unknown" in CONDITION_CHANNELS:
    raise RuntimeError("egis_cat_0_unknown must not be used for training")


In [ ]:
def parse_fill_info(z):
    if "fill_info" not in z.files:
        return {}
    raw = npz_scalar(z["fill_info"])
    if isinstance(raw, bytes):
        raw = raw.decode("utf-8")
    try:
        return json.loads(str(raw))
    except Exception:
        return {"raw": str(raw)}


def validate_samples(paths, limit=None):
    rows = []
    check_paths = paths if limit is None else paths[:limit]
    for path in tqdm(check_paths, desc="validate samples"):
        with np.load(path, allow_pickle=True) as z:
            condition_channels = [str(x) for x in z["condition_channels"]]
            target_channels = [str(x) for x in z["target_channels"]]
            target = z["target"].astype(np.float32)
            target_mask = z["target_mask"].astype(bool)
            cloud_fill_mask = z["cloud_fill_mask"].astype(bool)
            fill_info = parse_fill_info(z)
        if condition_channels != CONDITION_CHANNELS:
            raise RuntimeError(f"condition channel mismatch: {path.name}")
        if target_channels != TARGET_CHANNELS:
            raise RuntimeError(f"target channel mismatch: {path.name}: {target_channels}")
        if target.shape[0] != len(TARGET_CHANNELS):
            raise RuntimeError(f"target channel count mismatch: {path.name}")
        if target_mask.shape != target.shape:
            raise RuntimeError(f"target_mask shape mismatch: {path.name}")
        if not np.isfinite(target[target_mask]).all():
            raise RuntimeError(f"non-finite target inside target_mask: {path.name}")
        rows.append({
            "file": path.name,
            "split": path.parent.name,
            "target_valid_fraction": float(target_mask.mean()),
            "cloud_fill_fraction": float(cloud_fill_mask.mean()),
            "fill_method": fill_info.get("method", "missing"),
        })
    return pd.DataFrame(rows)

validation_df = validate_samples(all_paths)
validation_df.to_csv(RUN_DIR / "dataset_validation.csv", index=False, encoding="utf-8-sig")
print(validation_df.groupby(["split", "fill_method"]).size())
print(validation_df[["target_valid_fraction", "cloud_fill_fraction"]].describe())
if (validation_df["fill_method"] == "raw_unfilled").any():
    raise RuntimeError("raw_unfilled samples remain. Run global cloud-fill target generation before CVAE training.")


In [ ]:
def update_bounds(minima, maxima, values, mask=None):
    values = values.astype(np.float32)
    if mask is not None:
        values = np.where(mask, values, np.nan)
    finite = np.isfinite(values)
    for ch in range(values.shape[0]):
        vals = values[ch][finite[ch]]
        if vals.size:
            minima[ch] = min(minima[ch], float(vals.min()))
            maxima[ch] = max(maxima[ch], float(vals.max()))


def compute_stats(paths):
    c_ch = len(CONDITION_CHANNELS)
    t_ch = len(TARGET_CHANNELS)
    m_ch = len(META_CHANNELS)
    cmin = np.full(c_ch, np.inf, dtype=np.float64)
    cmax = np.full(c_ch, -np.inf, dtype=np.float64)
    tmin = np.full(t_ch, np.inf, dtype=np.float64)
    tmax = np.full(t_ch, -np.inf, dtype=np.float64)
    mmin = np.full(m_ch, np.inf, dtype=np.float64)
    mmax = np.full(m_ch, -np.inf, dtype=np.float64)
    for path in tqdm(paths, desc="stats"):
        with np.load(path, allow_pickle=True) as z:
            update_bounds(cmin, cmax, z["condition"].astype(np.float32))
            update_bounds(tmin, tmax, z["target"].astype(np.float32), z["target_mask"].astype(bool))
            meta = z["meta"].astype(np.float32)
            mmin = np.minimum(mmin, meta)
            mmax = np.maximum(mmax, meta)
    return {
        "condition_channels": CONDITION_CHANNELS,
        "target_channels": TARGET_CHANNELS,
        "meta_channels": META_CHANNELS,
        "condition": {name: {"min": float(cmin[i]), "max": float(cmax[i])} for i, name in enumerate(CONDITION_CHANNELS)},
        "target": {name: {"min": float(tmin[i]), "max": float(tmax[i])} for i, name in enumerate(TARGET_CHANNELS)},
        "meta": {name: {"min": float(mmin[i]), "max": float(mmax[i])} for i, name in enumerate(META_CHANNELS)},
    }


stats_path = RUN_DIR / "normalization_stats_lst_cvae_unet_skip_v25.json"
stats = compute_stats(train_paths)
stats_path.write_text(json.dumps(stats, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved stats:", stats_path)


In [ ]:
def minmax(values, channel_stats, names):
    out = values.astype(np.float32, copy=True)
    for idx, name in enumerate(names):
        lo = channel_stats[name]["min"]
        hi = channel_stats[name]["max"]
        denom = hi - lo
        out[idx] = 0.0 if abs(denom) <= 1e-12 else (out[idx] - lo) / denom
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def denormalize(values, channel_stats, names):
    out = values.astype(np.float32, copy=True)
    for idx, name in enumerate(names):
        lo = channel_stats[name]["min"]
        hi = channel_stats[name]["max"]
        out[idx] = out[idx] * (hi - lo) + lo
    return out


class PatchDataset(Dataset):
    def __init__(self, paths, stats):
        self.paths = list(paths)
        self.stats = stats

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        with np.load(self.paths[idx], allow_pickle=True) as sample:
            condition = minmax(sample["condition"], self.stats["condition"], CONDITION_CHANNELS)
            target = minmax(sample["target"], self.stats["target"], TARGET_CHANNELS)
            target_mask = sample["target_mask"].astype(np.float32)
            clear_mask = sample["clear_mask"].astype(np.float32)[None]
            cloud_fill_mask = sample["cloud_fill_mask"].astype(np.float32)[None]
            meta = sample["meta"].astype(np.float32)
        return {
            "condition": torch.from_numpy(condition),
            "target": torch.from_numpy(target),
            "target_mask": torch.from_numpy(target_mask),
            "clear_mask": torch.from_numpy(clear_mask),
            "cloud_fill_mask": torch.from_numpy(cloud_fill_mask),
            "meta": torch.from_numpy(meta),
            "path_index": torch.tensor(idx, dtype=torch.long),
        }


train_loader = DataLoader(PatchDataset(train_paths, stats), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(PatchDataset(val_paths, stats), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS) if val_paths else None
test_loader = DataLoader(PatchDataset(test_paths, stats), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS) if test_paths else None


In [ ]:
class ConvBlock(nn.Sequential):
    def __init__(self, in_ch, out_ch):
        super().__init__(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.GroupNorm(min(8, out_ch), out_ch),
            nn.SiLU(inplace=True),
        )


class DownBlock(nn.Sequential):
    def __init__(self, in_ch, out_ch):
        super().__init__(
            nn.Conv2d(in_ch, out_ch, 4, 2, 1),
            nn.GroupNorm(min(8, out_ch), out_ch),
            nn.SiLU(inplace=True),
        )


class UpBlock(nn.Sequential):
    def __init__(self, in_ch, out_ch):
        super().__init__(
            nn.ConvTranspose2d(in_ch, out_ch, 4, 2, 1),
            nn.GroupNorm(min(8, out_ch), out_ch),
            nn.SiLU(inplace=True),
        )


class ConditionEncoder(nn.Module):
    def __init__(self, condition_ch, width):
        super().__init__()
        self.stem = ConvBlock(condition_ch, width)          # 256x256
        self.down1 = DownBlock(width, width)               # 128x128
        self.down2 = DownBlock(width, width * 2)           # 64x64
        self.down3 = DownBlock(width * 2, width * 4)       # 32x32
        self.down4 = DownBlock(width * 4, width * 8)       # 16x16
        self.down5 = DownBlock(width * 8, width * 8)       # 8x8

    def forward(self, condition):
        s0 = self.stem(condition)
        s1 = self.down1(s0)
        s2 = self.down2(s1)
        s3 = self.down3(s2)
        s4 = self.down4(s3)
        bottleneck = self.down5(s4)
        skips = {"s0": s0, "s1": s1, "s2": s2, "s3": s3, "s4": s4}
        return bottleneck, skips


class PosteriorEncoder(nn.Module):
    def __init__(self, in_ch, width):
        super().__init__()
        self.net = nn.Sequential(
            ConvBlock(in_ch, width),
            DownBlock(width, width),
            DownBlock(width, width * 2),
            DownBlock(width * 2, width * 4),
            DownBlock(width * 4, width * 8),
            DownBlock(width * 8, width * 8),
        )

    def forward(self, condition, target):
        return self.net(torch.cat([condition, target], dim=1))


class EnvironmentalCVAEUNetSkip(nn.Module):
    def __init__(self, condition_ch, target_ch, meta_dim, latent_dim=64, width=32, grid_n=256):
        super().__init__()
        self.latent_dim = latent_dim
        self.width = width
        self.grid_n = grid_n
        reduced = grid_n // 32
        if reduced < 1 or grid_n % 32 != 0:
            raise ValueError(f"grid_n must be divisible by 32, got {grid_n}")
        self.reduced = reduced

        self.condition_encoder = ConditionEncoder(condition_ch, width)
        self.posterior_encoder = PosteriorEncoder(condition_ch + target_ch, width)

        flat = width * 8 * reduced * reduced
        self.posterior_head = nn.Sequential(nn.Linear(flat + meta_dim, width * 16), nn.SiLU(inplace=True))
        self.mu = nn.Linear(width * 16, latent_dim)
        self.logvar = nn.Linear(width * 16, latent_dim)
        self.latent_map = nn.Sequential(nn.Linear(latent_dim + meta_dim, flat), nn.SiLU(inplace=True))

        self.up1 = UpBlock(width * 16, width * 8)  # 8 -> 16
        self.fuse1 = ConvBlock(width * 16, width * 8)
        self.up2 = UpBlock(width * 8, width * 4)   # 16 -> 32
        self.fuse2 = ConvBlock(width * 8, width * 4)
        self.up3 = UpBlock(width * 4, width * 2)   # 32 -> 64
        self.fuse3 = ConvBlock(width * 4, width * 2)
        self.up4 = UpBlock(width * 2, width)       # 64 -> 128
        self.fuse4 = ConvBlock(width * 2, width)
        self.up5 = UpBlock(width, width)           # 128 -> 256
        self.fuse5 = ConvBlock(width * 2, width)
        self.out = nn.Conv2d(width, target_ch, 3, padding=1)

    def encode_condition(self, condition):
        return self.condition_encoder(condition)

    def encode(self, condition, target, meta):
        h = self.posterior_encoder(condition, target).flatten(1)
        h = self.posterior_head(torch.cat([h, meta], dim=1))
        return self.mu(h), self.logvar(h).clamp(-20.0, 10.0)

    def decode(self, condition, meta, latent):
        c, skips = self.encode_condition(condition)
        z = self.latent_map(torch.cat([latent, meta], dim=1)).view(
            condition.shape[0], self.width * 8, self.reduced, self.reduced
        )
        x = torch.cat([c, z], dim=1)
        x = self.up1(x)
        x = self.fuse1(torch.cat([x, skips["s4"]], dim=1))
        x = self.up2(x)
        x = self.fuse2(torch.cat([x, skips["s3"]], dim=1))
        x = self.up3(x)
        x = self.fuse3(torch.cat([x, skips["s2"]], dim=1))
        x = self.up4(x)
        x = self.fuse4(torch.cat([x, skips["s1"]], dim=1))
        x = self.up5(x)
        x = self.fuse5(torch.cat([x, skips["s0"]], dim=1))
        return self.out(x)

    def forward(self, condition, target, meta):
        mu, logvar = self.encode(condition, target, meta)
        latent = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
        return self.decode(condition, meta, latent), mu, logvar

    @torch.no_grad()
    def sample(self, condition, meta, latent=None):
        if latent is None:
            latent = torch.randn(condition.shape[0], self.latent_dim, device=condition.device)
        return self.decode(condition, meta, latent)


def masked_cvae_loss(pred, target, mask, mu, logvar, beta):
    mask = mask.to(pred.dtype)
    recon = ((pred - target).square() * mask).sum() / mask.sum().clamp_min(1.0)
    kl = -0.5 * torch.mean(torch.sum(1.0 + logvar - mu.square() - logvar.exp(), dim=1))
    return recon + beta * kl, recon, kl


with np.load(train_paths[0], allow_pickle=True) as z:
    grid_n = int(z["target"].shape[-1])

model = EnvironmentalCVAEUNetSkip(
    condition_ch=len(CONDITION_CHANNELS),
    target_ch=len(TARGET_CHANNELS),
    meta_dim=len(META_CHANNELS),
    latent_dim=LATENT_DIM,
    width=WIDTH,
    grid_n=grid_n,
).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=4)
print("model:", model.__class__.__name__)
print("params M:", sum(p.numel() for p in model.parameters()) / 1e6)


In [ ]:
def run_epoch(loader, training, beta_eff):
    model.train(training)
    totals = {"loss": 0.0, "recon": 0.0, "kl": 0.0, "n": 0}
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for batch in loader:
            condition = batch["condition"].to(DEVICE)
            target = batch["target"].to(DEVICE)
            target_mask = batch["target_mask"].to(DEVICE)
            meta = batch["meta"].to(DEVICE)
            pred, mu, logvar = model(condition, target, meta)
            loss, recon, kl = masked_cvae_loss(pred, target, target_mask, mu, logvar, beta_eff)
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()
            bs = condition.shape[0]
            totals["loss"] += float(loss.detach().cpu()) * bs
            totals["recon"] += float(recon.detach().cpu()) * bs
            totals["kl"] += float(kl.detach().cpu()) * bs
            totals["n"] += bs
    n = max(totals["n"], 1)
    return {key: totals[key] / n for key in ["loss", "recon", "kl"]}


def write_checkpoint_summary(rows):
    pd.DataFrame(rows).to_csv(RUN_DIR / "checkpoint_summary.csv", index=False, encoding="utf-8-sig")


history = []
checkpoint_rows = []
best_val = float("inf")
best_balanced_val = float("inf")
bad_epochs = 0
latest_path = RUN_DIR / "latest.pt"
best_val_path = RUN_DIR / "best_val.pt"
best_path = RUN_DIR / "best.pt"  # compatibility alias for best_val.pt
best_balanced_path = RUN_DIR / "best_balanced.pt"
checkpoint_dir = RUN_DIR / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

for epoch in range(1, EPOCHS + 1):
    beta_eff = BETA * min(1.0, epoch / KL_WARMUP_EPOCHS)
    train_metrics = run_epoch(train_loader, training=True, beta_eff=beta_eff)
    val_metrics = run_epoch(val_loader, training=False, beta_eff=beta_eff) if val_loader is not None else train_metrics
    scheduler.step(val_metrics["loss"])
    row = {
        "epoch": epoch,
        "beta_eff": beta_eff,
        "train_loss": train_metrics["loss"],
        "train_recon": train_metrics["recon"],
        "train_kl": train_metrics["kl"],
        "val_loss": val_metrics["loss"],
        "val_recon": val_metrics["recon"],
        "val_kl": val_metrics["kl"],
        "lr": optimizer.param_groups[0]["lr"],
    }
    history.append(row)
    history_df = pd.DataFrame(history)
    history_df.to_csv(RUN_DIR / "history.csv", index=False)

    checkpoint = {
        "epoch": epoch,
        "beta_eff": beta_eff,
        "model_name": model.__class__.__name__,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "normalization_stats": stats,
        "condition_channels": CONDITION_CHANNELS,
        "target_channels": TARGET_CHANNELS,
        "meta_channels": META_CHANNELS,
        "latent_dim": LATENT_DIM,
        "width": WIDTH,
        "beta": BETA,
        "kl_warmup_epochs": KL_WARMUP_EPOCHS,
        "balanced_min_kl": BALANCED_MIN_KL,
        "grid_n": grid_n,
        "metrics": row,
    }
    torch.save(checkpoint, latest_path)
    if SAVE_EVERY_EPOCH:
        torch.save(checkpoint, checkpoint_dir / f"epoch_{epoch:03d}.pt")

    saved_tags = ["latest"]
    if val_metrics["loss"] < best_val:
        best_val = val_metrics["loss"]
        bad_epochs = 0
        torch.save(checkpoint, best_val_path)
        torch.save(checkpoint, best_path)
        saved_tags.extend(["best_val", "best"])
    else:
        bad_epochs += 1

    if val_metrics["kl"] >= BALANCED_MIN_KL and val_metrics["loss"] < best_balanced_val:
        best_balanced_val = val_metrics["loss"]
        torch.save(checkpoint, best_balanced_path)
        saved_tags.append("best_balanced")

    checkpoint_rows.append({
        "epoch": epoch,
        "saved_tags": ",".join(saved_tags),
        "val_loss": val_metrics["loss"],
        "val_kl": val_metrics["kl"],
        "train_loss": train_metrics["loss"],
        "train_kl": train_metrics["kl"],
        "best_val_epoch": int(history_df.loc[history_df["val_loss"].idxmin(), "epoch"]),
        "best_val_loss": float(history_df["val_loss"].min()),
        "best_val_kl": float(history_df.loc[history_df["val_loss"].idxmin(), "val_kl"]),
        "best_balanced_min_kl": BALANCED_MIN_KL,
    })
    write_checkpoint_summary(checkpoint_rows)

    print(
        f"{epoch:03d} beta={beta_eff:.2e} train={train_metrics['loss']:.5f} recon={train_metrics['recon']:.5f} kl={train_metrics['kl']:.5f} "
        f"val={val_metrics['loss']:.5f} val_kl={val_metrics['kl']:.5f} lr={optimizer.param_groups[0]['lr']:.2e} saved={','.join(saved_tags)}"
    )
    if bad_epochs >= PATIENCE:
        print("early stop")
        break

(RUN_DIR / "metrics.json").write_text(json.dumps(history, ensure_ascii=False, indent=2), encoding="utf-8")
print("best_val:", best_val_path)
print("best_balanced:", best_balanced_path if best_balanced_path.exists() else "not saved")
print("checkpoint_summary:", RUN_DIR / "checkpoint_summary.csv")


In [ ]:
CHECKPOINT_PATH = RUN_DIR / "best_val.pt"
if not CHECKPOINT_PATH.exists():
    CHECKPOINT_PATH = RUN_DIR / "best.pt"
if not CHECKPOINT_PATH.exists():
    CHECKPOINT_PATH = RUN_DIR / "latest.pt"
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["model"])
model.eval()
print("loaded:", CHECKPOINT_PATH)
print("epoch:", checkpoint.get("epoch"))
print("metrics:", checkpoint.get("metrics"))


## Checkpoint selection

`best_val`의 KL과 `best_balanced` 후보를 바로 확인합니다.


In [ ]:
summary_path = RUN_DIR / "checkpoint_summary.csv"
if summary_path.exists():
    ckpt_summary = pd.read_csv(summary_path)
    display(ckpt_summary.tail(10))
    best_val_row = ckpt_summary.loc[ckpt_summary["best_val_loss"].idxmin()]
    print("current best_val epoch/loss/kl:", int(best_val_row["best_val_epoch"]), best_val_row["best_val_loss"], best_val_row["best_val_kl"])
    if best_balanced_path.exists():
        balanced = torch.load(best_balanced_path, map_location="cpu")
        print("best_balanced epoch/metrics:", balanced.get("epoch"), balanced.get("metrics"))
else:
    print("checkpoint_summary.csv not found yet")


In [ ]:
@torch.no_grad()
def evaluate(paths, loader, split_name):
    if loader is None:
        return pd.DataFrame()
    rows = []
    pred_dir = RUN_DIR / "predictions" / split_name
    pred_dir.mkdir(parents=True, exist_ok=True)
    sample_idx = 0
    for batch in loader:
        condition = batch["condition"].to(DEVICE)
        target = batch["target"].to(DEVICE)
        target_mask = batch["target_mask"].to(DEVICE)
        meta = batch["meta"].to(DEVICE)
        pred, mu, logvar = model(condition, target, meta)
        pred_np = pred.detach().cpu().numpy()
        target_np = target.detach().cpu().numpy()
        mask_np = target_mask.detach().cpu().numpy().astype(bool)
        clear_np = batch["clear_mask"].numpy().astype(bool)
        cloud_np = batch["cloud_fill_mask"].numpy().astype(bool)
        for b in range(pred_np.shape[0]):
            sample_path = paths[sample_idx]
            pred_denorm = denormalize(pred_np[b], stats["target"], TARGET_CHANNELS)
            target_denorm = denormalize(target_np[b], stats["target"], TARGET_CHANNELS)
            diff = pred_denorm[0] - target_denorm[0]
            for region_name, region_mask in [
                ("all", mask_np[b, 0]),
                ("landsat_clear", clear_np[b, 0] & mask_np[b, 0]),
                ("cloud_filled", cloud_np[b, 0] & mask_np[b, 0]),
            ]:
                valid = region_mask & np.isfinite(diff)
                if valid.any():
                    rows.append({
                        "split": split_name,
                        "file": sample_path.name,
                        "region": region_name,
                        "channel": TARGET_CHANNELS[0],
                        "rmse_k": float(np.sqrt(np.mean(diff[valid] ** 2))),
                        "mae_k": float(np.mean(np.abs(diff[valid]))),
                        "bias_k": float(np.mean(diff[valid])),
                        "valid_fraction": float(valid.mean()),
                    })
            np.savez_compressed(
                pred_dir / sample_path.name,
                prediction=pred_denorm,
                target=target_denorm,
                target_mask=mask_np[b],
                clear_mask=clear_np[b, 0],
                cloud_fill_mask=cloud_np[b, 0],
                target_channels=np.array(TARGET_CHANNELS),
            )
            sample_idx += 1
    return pd.DataFrame(rows)


metrics = pd.concat([
    evaluate(val_paths, val_loader, "val"),
    evaluate(test_paths, test_loader, "test"),
], ignore_index=True)
metrics.to_csv(RUN_DIR / "metrics_detail.csv", index=False)
summary = metrics.groupby(["split", "region", "channel"])[["rmse_k", "mae_k", "bias_k", "valid_fraction"]].mean().reset_index()
summary.to_csv(RUN_DIR / "metrics_summary.csv", index=False)
summary


## Condition-only evaluation

Target을 posterior encoder에 넣지 않고, condition/meta만으로 예측 성능을 평가합니다. `prior_mean`은 z=0, `prior_ensemble`은 prior sample 여러 개 평균입니다.


In [ ]:
@torch.no_grad()
def evaluate_condition_only(paths, loader, split_name, mode="prior_mean", n_samples=8):
    if loader is None:
        return pd.DataFrame()
    if mode not in {"prior_mean", "prior_ensemble"}:
        raise ValueError("mode must be 'prior_mean' or 'prior_ensemble'")
    rows = []
    pred_dir = RUN_DIR / "predictions_condition_only" / mode / split_name
    pred_dir.mkdir(parents=True, exist_ok=True)
    sample_idx = 0
    for batch in loader:
        condition = batch["condition"].to(DEVICE)
        target = batch["target"].to(DEVICE)
        target_mask = batch["target_mask"].to(DEVICE)
        meta = batch["meta"].to(DEVICE)
        if mode == "prior_mean":
            latent = torch.zeros(condition.shape[0], LATENT_DIM, device=DEVICE)
            pred = model.decode(condition, meta, latent)
        else:
            preds = [model.sample(condition, meta) for _ in range(n_samples)]
            pred = torch.stack(preds, dim=0).mean(dim=0)
        pred_np = pred.detach().cpu().numpy()
        target_np = target.detach().cpu().numpy()
        mask_np = target_mask.detach().cpu().numpy().astype(bool)
        clear_np = batch["clear_mask"].numpy().astype(bool)
        cloud_np = batch["cloud_fill_mask"].numpy().astype(bool)
        for b in range(pred_np.shape[0]):
            sample_path = paths[sample_idx]
            pred_denorm = denormalize(pred_np[b], stats["target"], TARGET_CHANNELS)
            target_denorm = denormalize(target_np[b], stats["target"], TARGET_CHANNELS)
            diff = pred_denorm[0] - target_denorm[0]
            for region_name, region_mask in [
                ("all", mask_np[b, 0]),
                ("landsat_clear", clear_np[b, 0] & mask_np[b, 0]),
                ("cloud_filled", cloud_np[b, 0] & mask_np[b, 0]),
            ]:
                valid = region_mask & np.isfinite(diff)
                if valid.any():
                    rows.append({
                        "split": split_name,
                        "mode": mode,
                        "file": sample_path.name,
                        "region": region_name,
                        "channel": TARGET_CHANNELS[0],
                        "rmse_k": float(np.sqrt(np.mean(diff[valid] ** 2))),
                        "mae_k": float(np.mean(np.abs(diff[valid]))),
                        "bias_k": float(np.mean(diff[valid])),
                        "valid_fraction": float(valid.mean()),
                    })
            np.savez_compressed(
                pred_dir / sample_path.name,
                prediction=pred_denorm,
                target=target_denorm,
                target_mask=mask_np[b],
                clear_mask=clear_np[b, 0],
                cloud_fill_mask=cloud_np[b, 0],
                target_channels=np.array(TARGET_CHANNELS),
                mode=np.array(mode),
                n_samples=np.array(n_samples if mode == "prior_ensemble" else 1),
            )
            sample_idx += 1
    return pd.DataFrame(rows)

condition_only_metrics = pd.concat([
    evaluate_condition_only(val_paths, val_loader, "val", mode="prior_mean"),
    evaluate_condition_only(test_paths, test_loader, "test", mode="prior_mean"),
    evaluate_condition_only(val_paths, val_loader, "val", mode="prior_ensemble", n_samples=8),
    evaluate_condition_only(test_paths, test_loader, "test", mode="prior_ensemble", n_samples=8),
], ignore_index=True)
condition_only_metrics.to_csv(RUN_DIR / "metrics_condition_only_detail.csv", index=False)
condition_only_summary = condition_only_metrics.groupby(["split", "mode", "region", "channel"])[["rmse_k", "mae_k", "bias_k", "valid_fraction"]].mean().reset_index()
condition_only_summary.to_csv(RUN_DIR / "metrics_condition_only_summary.csv", index=False)
condition_only_summary


## Prior ensemble visualization

Condition/meta만 넣고 prior latent를 여러 번 샘플링한 결과를 개별 샘플, 평균, 표준편차로 확인합니다.


In [ ]:
@torch.no_grad()
def plot_prior_ensemble_sample(sample_path=None, split="test", n_samples=8, seed=20260522):
    if sample_path is None:
        candidate_paths = test_paths if split == "test" else val_paths
        if not candidate_paths:
            raise ValueError(f"no paths for split={split}")
        sample_path = candidate_paths[0]
    sample_path = Path(sample_path)
    dataset = PatchDataset([sample_path], stats)
    batch = next(iter(DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)))

    condition = batch["condition"].to(DEVICE)
    target = batch["target"].to(DEVICE)
    meta = batch["meta"].to(DEVICE)
    target_mask = batch["target_mask"].numpy()[0, 0].astype(bool)
    clear_mask = batch["clear_mask"].numpy()[0, 0].astype(bool)
    cloud_mask = batch["cloud_fill_mask"].numpy()[0, 0].astype(bool)

    generator = torch.Generator(device=DEVICE)
    generator.manual_seed(seed)
    preds = []
    for _ in range(n_samples):
        latent = torch.randn(condition.shape[0], LATENT_DIM, generator=generator, device=DEVICE)
        pred = model.decode(condition, meta, latent)
        preds.append(pred.detach().cpu().numpy()[0])
    preds = np.stack(preds, axis=0)

    pred_denorm = np.stack([denormalize(preds[i], stats["target"], TARGET_CHANNELS)[0] for i in range(n_samples)], axis=0)
    target_denorm = denormalize(target.detach().cpu().numpy()[0], stats["target"], TARGET_CHANNELS)[0]
    ens_mean = pred_denorm.mean(axis=0)
    ens_std = pred_denorm.std(axis=0)
    error = ens_mean - target_denorm

    valid_error = np.where(target_mask, error, np.nan)
    vmin = np.nanpercentile(target_denorm[target_mask], 2)
    vmax = np.nanpercentile(target_denorm[target_mask], 98)
    err_abs = np.nanpercentile(np.abs(valid_error), 98)
    std_max = np.nanpercentile(ens_std[target_mask], 98)

    fig, axes = plt.subplots(2, 4, figsize=(18, 9), constrained_layout=True)
    axes = axes.ravel()
    panels = [
        (target_denorm, "target lst_k_filled", "turbo", vmin, vmax),
        (ens_mean, f"prior ensemble mean n={n_samples}", "turbo", vmin, vmax),
        (ens_std, "ensemble std", "magma", 0, std_max),
        (valid_error, "mean - target", "coolwarm", -err_abs, err_abs),
        (pred_denorm[0], "sample 1", "turbo", vmin, vmax),
        (pred_denorm[min(1, n_samples - 1)], "sample 2", "turbo", vmin, vmax),
        (pred_denorm[min(2, n_samples - 1)], "sample 3", "turbo", vmin, vmax),
        (pred_denorm[min(3, n_samples - 1)], "sample 4", "turbo", vmin, vmax),
    ]
    for ax, (arr, title, cmap, lo, hi) in zip(axes, panels):
        im = ax.imshow(arr, cmap=cmap, vmin=lo, vmax=hi)
        ax.set_title(title)
        ax.axis("off")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    rmse_all = float(np.sqrt(np.nanmean(valid_error[target_mask] ** 2)))
    mae_all = float(np.nanmean(np.abs(valid_error[target_mask])))
    rmse_cloud = float(np.sqrt(np.nanmean(valid_error[cloud_mask & target_mask] ** 2))) if (cloud_mask & target_mask).any() else np.nan
    rmse_clear = float(np.sqrt(np.nanmean(valid_error[clear_mask & target_mask] ** 2))) if (clear_mask & target_mask).any() else np.nan
    fig.suptitle(
        f"{sample_path.name}\nRMSE all={rmse_all:.2f}K, MAE all={mae_all:.2f}K, "
        f"cloud={rmse_cloud:.2f}K, clear={rmse_clear:.2f}K",
        fontsize=12,
    )
    out_path = RUN_DIR / "prior_ensemble_visualization.png"
    fig.savefig(out_path, dpi=160)
    plt.show()
    print("saved:", out_path)
    return {
        "sample_path": sample_path,
        "rmse_all": rmse_all,
        "mae_all": mae_all,
        "rmse_cloud": rmse_cloud,
        "rmse_clear": rmse_clear,
        "ensemble_std_mean": float(np.nanmean(ens_std[target_mask])),
        "ensemble_std_p95": float(np.nanpercentile(ens_std[target_mask], 95)),
    }

plot_prior_ensemble_sample(split="test", n_samples=8)


In [ ]:
@torch.no_grad()
def decode_with_posterior_mean(sample_path):
    one_loader = DataLoader(PatchDataset([sample_path], stats), batch_size=1, shuffle=False, num_workers=0)
    batch = next(iter(one_loader))
    condition = batch["condition"].to(DEVICE)
    target = batch["target"].to(DEVICE)
    meta = batch["meta"].to(DEVICE)
    mu, logvar = model.encode(condition, target, meta)
    pred = model.decode(condition, meta, mu)
    pred_denorm = denormalize(pred.detach().cpu().numpy()[0], stats["target"], TARGET_CHANNELS)
    target_denorm = denormalize(target.detach().cpu().numpy()[0], stats["target"], TARGET_CHANNELS)
    mask = batch["target_mask"].numpy()[0].astype(bool)
    return pred_denorm, target_denorm, mask


def plot_prediction_sample(split_name="test", sample_number=0):
    split_paths = sorted((SAMPLE_DIR / split_name).glob("*.npz"))
    if not split_paths:
        raise FileNotFoundError(f"no samples for {split_name}")
    sample_path = split_paths[sample_number]
    pred, target, mask = decode_with_posterior_mean(sample_path)
    truth = np.where(mask[0], target[0], np.nan)
    estimate = np.where(mask[0], pred[0], np.nan)
    error = estimate - truth
    vmin, vmax = np.nanpercentile(truth, [2, 98])
    err_lim = np.nanpercentile(np.abs(error), 98)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
    axes[0].imshow(truth, cmap="turbo", vmin=vmin, vmax=vmax)
    axes[0].set_title("target lst_k_filled")
    axes[1].imshow(estimate, cmap="turbo", vmin=vmin, vmax=vmax)
    axes[1].set_title("CVAE posterior mean")
    axes[2].imshow(error, cmap="coolwarm", vmin=-err_lim, vmax=err_lim)
    axes[2].set_title("error K")
    for ax in axes:
        ax.axis("off")
    fig.suptitle(f"{split_name}: {sample_path.name}")
    out = RUN_DIR / f"prediction_preview_{split_name}_{sample_number}.png"
    fig.savefig(out, dpi=160)
    plt.show()
    print("saved:", out)


plot_prediction_sample("test" if test_paths else "val", 0)


In [ ]:
@torch.no_grad()
def sample_prior_variants(sample_path, n=4):
    one_loader = DataLoader(PatchDataset([sample_path], stats), batch_size=1, shuffle=False, num_workers=0)
    batch = next(iter(one_loader))
    condition = batch["condition"].to(DEVICE)
    meta = batch["meta"].to(DEVICE)
    variants = []
    for _ in range(n):
        pred = model.sample(condition, meta).detach().cpu().numpy()[0]
        variants.append(denormalize(pred, stats["target"], TARGET_CHANNELS)[0])
    return variants


sample_path = (test_paths or val_paths or train_paths)[0]
variants = sample_prior_variants(sample_path, n=4)
fig, axes = plt.subplots(1, len(variants), figsize=(4 * len(variants), 4), constrained_layout=True)
for ax, arr, idx in zip(np.ravel(axes), variants, range(len(variants))):
    ax.imshow(arr, cmap="turbo")
    ax.set_title(f"prior sample {idx + 1}")
    ax.axis("off")
out = RUN_DIR / "prior_samples_preview.png"
fig.savefig(out, dpi=160)
plt.show()
print("saved:", out)


## City prior ensemble visualizations

서울, 인천 및 내륙 도시 샘플을 prior ensemble 방식으로 시각화합니다. 각 도시는 test -> val -> train 순서로 샘플을 찾습니다.


In [ ]:
CITY_VIS_CITIES = [
    "seoul",
    "incheon",
    "suwon",
    "daejeon",
    "daegu",
    "cheongju",
    "chuncheon",
    "gwangju",
    "jeonju",
]
CITY_VIS_SPLIT_PRIORITY = ["test", "val", "train"]
CITY_VIS_N_SAMPLES = 8
CITY_VIS_SEED = 20260522


def paths_for_split(split):
    if split == "train":
        return train_paths
    if split == "val":
        return val_paths
    if split == "test":
        return test_paths
    raise ValueError(split)


def find_city_sample(city, split_priority=CITY_VIS_SPLIT_PRIORITY):
    city = city.lower()
    for split in split_priority:
        hits = [p for p in paths_for_split(split) if p.name.lower().startswith(city + "_")]
        if hits:
            return split, hits[0]
    return None, None


@torch.no_grad()
def prior_ensemble_arrays(sample_path, n_samples=CITY_VIS_N_SAMPLES, seed=CITY_VIS_SEED):
    sample_path = Path(sample_path)
    dataset = PatchDataset([sample_path], stats)
    batch = next(iter(DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)))
    condition = batch["condition"].to(DEVICE)
    target = batch["target"].to(DEVICE)
    meta = batch["meta"].to(DEVICE)
    target_mask = batch["target_mask"].numpy()[0, 0].astype(bool)
    clear_mask = batch["clear_mask"].numpy()[0, 0].astype(bool)
    cloud_mask = batch["cloud_fill_mask"].numpy()[0, 0].astype(bool)

    generator = torch.Generator(device=DEVICE)
    generator.manual_seed(seed)
    preds = []
    for _ in range(n_samples):
        latent = torch.randn(condition.shape[0], LATENT_DIM, generator=generator, device=DEVICE)
        pred = model.decode(condition, meta, latent)
        preds.append(pred.detach().cpu().numpy()[0])
    preds = np.stack(preds, axis=0)
    pred_k = np.stack([denormalize(preds[i], stats["target"], TARGET_CHANNELS)[0] for i in range(n_samples)], axis=0)
    target_k = denormalize(target.detach().cpu().numpy()[0], stats["target"], TARGET_CHANNELS)[0]
    mean_k = pred_k.mean(axis=0)
    std_k = pred_k.std(axis=0)
    error_k = mean_k - target_k
    return {
        "prediction_samples": pred_k,
        "prediction_mean": mean_k,
        "prediction_std": std_k,
        "target": target_k,
        "error": error_k,
        "target_mask": target_mask,
        "clear_mask": clear_mask,
        "cloud_mask": cloud_mask,
    }


def plot_city_prior_ensemble(city, split=None, sample_path=None, n_samples=CITY_VIS_N_SAMPLES, seed=CITY_VIS_SEED):
    if sample_path is None:
        if split is None:
            split, sample_path = find_city_sample(city)
        else:
            hits = [p for p in paths_for_split(split) if p.name.lower().startswith(city.lower() + "_")]
            sample_path = hits[0] if hits else None
    if sample_path is None:
        print(f"[missing] {city}: no sample found")
        return None

    arrays = prior_ensemble_arrays(sample_path, n_samples=n_samples, seed=seed)
    target = arrays["target"]
    mean = arrays["prediction_mean"]
    std = arrays["prediction_std"]
    err = arrays["error"]
    mask = arrays["target_mask"]
    clear = arrays["clear_mask"]
    cloud = arrays["cloud_mask"]
    valid = mask & np.isfinite(err)
    cloud_valid = cloud & valid
    clear_valid = clear & valid

    rmse = float(np.sqrt(np.mean(err[valid] ** 2)))
    mae = float(np.mean(np.abs(err[valid])))
    bias = float(np.mean(err[valid]))
    rmse_cloud = float(np.sqrt(np.mean(err[cloud_valid] ** 2))) if cloud_valid.any() else np.nan
    rmse_clear = float(np.sqrt(np.mean(err[clear_valid] ** 2))) if clear_valid.any() else np.nan

    vmin = float(np.nanpercentile(target[valid], 2))
    vmax = float(np.nanpercentile(target[valid], 98))
    err_abs = float(np.nanpercentile(np.abs(err[valid]), 98))
    std_max = float(np.nanpercentile(std[valid], 98))

    fig, axes = plt.subplots(1, 5, figsize=(22, 4.8), constrained_layout=True)
    panels = [
        (target, "target lst_k_filled", "turbo", vmin, vmax),
        (mean, f"prior ensemble mean n={n_samples}", "turbo", vmin, vmax),
        (np.where(valid, err, np.nan), "mean - target", "coolwarm", -err_abs, err_abs),
        (std, "ensemble std", "magma", 0.0, std_max),
        (cloud.astype(float), "cloud_fill_mask", "gray", 0.0, 1.0),
    ]
    for ax, (arr, title, cmap, lo, hi) in zip(axes, panels):
        im = ax.imshow(arr, cmap=cmap, vmin=lo, vmax=hi)
        ax.set_title(title)
        ax.axis("off")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    city_name = city.lower()
    title_split = split if split is not None else "auto"
    fig.suptitle(
        f"{city_name} | {title_split} | {Path(sample_path).name}\n"
        f"RMSE={rmse:.2f}K MAE={mae:.2f}K bias={bias:.2f}K cloud={rmse_cloud:.2f}K clear={rmse_clear:.2f}K",
        fontsize=11,
    )
    out_dir = RUN_DIR / "city_prior_ensemble_visualizations"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{city_name}_{Path(sample_path).stem}_prior_ensemble.png"
    fig.savefig(out_path, dpi=160)
    plt.show()
    print("saved:", out_path)
    return {
        "city": city_name,
        "split": title_split,
        "file": Path(sample_path).name,
        "rmse_k": rmse,
        "mae_k": mae,
        "bias_k": bias,
        "rmse_cloud_k": rmse_cloud,
        "rmse_clear_k": rmse_clear,
        "path": str(out_path),
    }


city_rows = []
for city in CITY_VIS_CITIES:
    row = plot_city_prior_ensemble(city, n_samples=CITY_VIS_N_SAMPLES, seed=CITY_VIS_SEED)
    if row is not None:
        city_rows.append(row)

city_visualization_summary = pd.DataFrame(city_rows)
city_visualization_summary.to_csv(RUN_DIR / "city_prior_ensemble_visualization_summary.csv", index=False, encoding="utf-8-sig")
city_visualization_summary
